# ============================================================
#  CONFIGURATION — edit these values before running
# ============================================================

In [ ]:
NYC_LIMIT       = 20    # datasets from NYC Open Data
DATA_GOV_LIMIT  = 20    # datasets from Data.gov

HDFS_BASE = "hdfs:///user/autoddg/pipeline"
LOCAL_OUTPUT_PARQUET = "/home/autoddg/outputs/descriptions.parquet"

MODEL_NAME          = "gpt-4o-mini"
MAX_SAMPLE_ROWS     = 5
REQUEST_TIMEOUT     = 20
MAX_ELAPSED_SECONDS = 30
MAX_CHARS           = 8000
MAX_LINES           = 8

# Derived HDFS paths
HDFS_METADATA = f"{HDFS_BASE}/step1_metadata"
HDFS_SAMPLES  = f"{HDFS_BASE}/step2_samples"
HDFS_PREPARED = f"{HDFS_BASE}/step3_prepared"
HDFS_EVAL     = f"{HDFS_BASE}/step5_evaluation"

print(f"NYC datasets      : {NYC_LIMIT}")
print(f"Data.gov datasets : {DATA_GOV_LIMIT}")
print(f"Total requested   : {NYC_LIMIT + DATA_GOV_LIMIT}")
print(f"Model             : {MODEL_NAME}")
print(f"HDFS base         : {HDFS_BASE}")
print(f"Local output      : {LOCAL_OUTPUT_PARQUET}")


In [ ]:
from __future__ import annotations

import csv
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from openai import OpenAI
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from autoddg import AutoDDG

csv.field_size_limit(sys.maxsize)
print("Imports OK.")


In [ ]:
spark = SparkSession.builder.appName("autoddg_pipeline").getOrCreate()
print(spark)


In [ ]:
NYC_CATALOG_URL     = "https://data.cityofnewyork.us/api/views.json"
NYC_VIEW_URL_TMPL   = "https://data.cityofnewyork.us/api/views/{dataset_id}.json"
DATA_GOV_SEARCH_URL = "https://catalog.data.gov/search"


def safe_get(dct: Dict[str, Any], key: str, default=None):
    return dct[key] if key in dct else default


def fetch_json(
    url: str,
    params: Optional[Dict[str, Any]] = None,
    retries: int = 3,
    sleep_seconds: float = 2.0,
) -> Any:
    last_error: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                url, params=params, timeout=60,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_error = e
            print(f"[WARN] fetch failed attempt={attempt}/{retries} url={url}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)
    raise RuntimeError(f"Failed to fetch URL after {retries} attempts: {url}") from last_error


def extract_keywords(category: Optional[str], tags: Optional[List[str]]) -> List[str]:
    out: List[str] = []
    if category:
        out.append(str(category))
    if tags:
        out.extend([str(t) for t in tags if t is not None])
    seen = set()
    deduped: List[str] = []
    for x in out:
        if x not in seen:
            seen.add(x)
            deduped.append(x)
    return deduped


def extract_column_names(columns: Optional[List[Dict[str, Any]]]) -> List[str]:
    names: List[str] = []
    for col in columns or []:
        name = safe_get(col, "name")
        if name:
            names.append(str(name))
    return names


def extract_column_types(columns: Optional[List[Dict[str, Any]]]) -> List[str]:
    types_: List[str] = []
    for col in columns or []:
        dtype = safe_get(col, "dataTypeName")
        types_.append(str(dtype) if dtype is not None else "unknown")
    return types_


def normalize_nyc_dataset(summary_entry: Dict[str, Any], detail_entry: Dict[str, Any]) -> Dict[str, Any]:
    dataset_id      = safe_get(summary_entry, "id")
    title           = safe_get(summary_entry, "name")
    description     = safe_get(summary_entry, "description")
    category        = safe_get(summary_entry, "category")
    tags            = safe_get(summary_entry, "tags", [])
    rows_updated_at = safe_get(summary_entry, "rowsUpdatedAt")
    detail_columns  = safe_get(detail_entry, "columns", [])
    license_info    = safe_get(detail_entry, "license")
    license_name    = None
    if isinstance(license_info, dict):
        license_name = safe_get(license_info, "name")
    return {
        "dataset_id":            str(dataset_id) if dataset_id is not None else None,
        "source":                "nyc_open_data",
        "title":                 str(title) if title is not None else None,
        "original_description":  str(description) if description is not None else None,
        "keywords_json":         json.dumps(extract_keywords(category, tags), ensure_ascii=False),
        "column_names_json":     json.dumps(extract_column_names(detail_columns), ensure_ascii=False),
        "column_types_raw_json": json.dumps(extract_column_types(detail_columns), ensure_ascii=False),
        "download_url":          f"https://data.cityofnewyork.us/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD" if dataset_id else None,
        "landing_page_url":      f"https://data.cityofnewyork.us/d/{dataset_id}" if dataset_id else None,
        "record_count_estimate": None,
        "last_updated":          str(rows_updated_at) if rows_updated_at is not None else None,
        "license":               str(license_name) if license_name is not None else None,
        "sample_rows_json":      None,
        "raw_metadata_json":     json.dumps(
            {"summary_entry": summary_entry, "detail_entry": detail_entry},
            ensure_ascii=False),
    }


def fetch_nyc_metadata(limit: int) -> List[Dict[str, Any]]:
    payload = fetch_json(NYC_CATALOG_URL)
    if not isinstance(payload, list):
        raise ValueError("Expected NYC catalog response to be a list.")
    trimmed = payload[:limit]
    rows: List[Dict[str, Any]] = []
    for idx, entry in enumerate(trimmed, start=1):
        dataset_id = safe_get(entry, "id")
        if not dataset_id:
            continue
        print(f"[INFO] NYC detail {idx}/{len(trimmed)} dataset_id={dataset_id}")
        detail = fetch_json(NYC_VIEW_URL_TMPL.format(dataset_id=dataset_id))
        rows.append(normalize_nyc_dataset(entry, detail))
    return rows


def normalize_data_gov_dataset(entry: Dict[str, Any]) -> Dict[str, Any]:
    dcat         = entry.get("dcat", {}) or {}
    dataset_id   = entry.get("identifier") or dcat.get("identifier")
    title        = entry.get("title") or dcat.get("title")
    notes        = entry.get("description") or dcat.get("description")
    tags         = entry.get("keyword") or dcat.get("keyword") or []
    modified     = dcat.get("modified")
    license_url  = dcat.get("license")
    landing_page = dcat.get("landingPage")
    distributions = dcat.get("distribution", []) or []
    csv_url = None
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        access_url = dist.get("accessURL") or dist.get("downloadURL")
        media_type = (dist.get("mediaType") or "").lower()
        fmt        = (dist.get("format") or "").lower()
        if access_url and (
            "csv" in media_type or "csv" in fmt
            or str(access_url).lower().endswith(".csv")
        ):
            csv_url = access_url
            break
    return {
        "dataset_id":            str(dataset_id) if dataset_id is not None else None,
        "source":                "data_gov",
        "title":                 str(title) if title is not None else None,
        "original_description":  str(notes) if notes is not None else None,
        "keywords_json":         json.dumps(
            [str(x) for x in tags] if isinstance(tags, list) else [], ensure_ascii=False),
        "column_names_json":     json.dumps([], ensure_ascii=False),
        "column_types_raw_json": json.dumps([], ensure_ascii=False),
        "download_url":          csv_url,
        "landing_page_url":      landing_page,
        "record_count_estimate": None,
        "last_updated":          str(modified) if modified is not None else None,
        "license":               str(license_url) if license_url is not None else None,
        "sample_rows_json":      None,
        "raw_metadata_json":     json.dumps(entry, ensure_ascii=False),
    }


def fetch_data_gov_metadata(limit: int) -> List[Dict[str, Any]]:
    params  = {"q": "", "per_page": limit}
    payload = fetch_json(DATA_GOV_SEARCH_URL, params=params)
    if not isinstance(payload, dict) or "results" not in payload:
        raise ValueError("Expected Data.gov search response with a 'results' key.")
    results = payload["results"]
    if not isinstance(results, list):
        raise ValueError("Expected Data.gov 'results' to be a list.")
    return [normalize_data_gov_dataset(entry) for entry in results[:limit]]


def build_metadata_schema() -> T.StructType:
    return T.StructType([
        T.StructField("dataset_id",            T.StringType(), True),
        T.StructField("source",                T.StringType(), True),
        T.StructField("title",                 T.StringType(), True),
        T.StructField("original_description",  T.StringType(), True),
        T.StructField("keywords_json",         T.StringType(), True),
        T.StructField("column_names_json",     T.StringType(), True),
        T.StructField("column_types_raw_json", T.StringType(), True),
        T.StructField("download_url",          T.StringType(), True),
        T.StructField("landing_page_url",      T.StringType(), True),
        T.StructField("record_count_estimate", T.LongType(),   True),
        T.StructField("last_updated",          T.StringType(), True),
        T.StructField("license",               T.StringType(), True),
        T.StructField("sample_rows_json",      T.StringType(), True),
        T.StructField("raw_metadata_json",     T.StringType(), True),
    ])


In [ ]:
print(f"[INFO] Fetching NYC metadata: {NYC_LIMIT}")
nyc_rows = fetch_nyc_metadata(limit=NYC_LIMIT)

print(f"[INFO] Fetching Data.gov metadata: {DATA_GOV_LIMIT}")
data_gov_rows = fetch_data_gov_metadata(limit=DATA_GOV_LIMIT)

all_rows = nyc_rows + data_gov_rows
if not all_rows:
    raise ValueError("No metadata rows were fetched from either source.")

metadata_df = spark.createDataFrame(all_rows, schema=build_metadata_schema())

print(f"[INFO] Writing {metadata_df.count()} combined rows to {HDFS_METADATA}")
metadata_df.write.mode("overwrite").parquet(HDFS_METADATA)

metadata_df.groupBy("source").count().show(truncate=False)
metadata_df.select("dataset_id", "source", "title", "download_url").show(20, truncate=False)


In [ ]:
def fetch_csv_sample_text(
    download_url: str,
    max_rows: int = 5,
    retries: int = 3,
    sleep_seconds: float = 2.0,
    request_timeout: int = 20,
    max_elapsed_seconds: int = 30,
) -> Optional[str]:
    last_error: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        start_time = time.time()
        try:
            with requests.get(
                download_url,
                timeout=request_timeout,
                stream=True,
                headers={"User-Agent": "Mozilla/5.0"},
            ) as response:
                response.raise_for_status()
                lines: List[str] = []
                for line in response.iter_lines(decode_unicode=True):
                    if time.time() - start_time > max_elapsed_seconds:
                        raise TimeoutError(f"sample fetch exceeded {max_elapsed_seconds}s")
                    if line is None:
                        continue
                    line = line.strip()
                    if not line:
                        continue
                    lines.append(line)
                    if len(lines) >= max_rows + 1:
                        break
                if not lines:
                    return None
                reader    = csv.reader(io.StringIO("\n".join(lines)))
                rows      = list(reader)
                if not rows:
                    return None
                header    = rows[0]
                data_rows = rows[1:]
                output    = io.StringIO()
                writer    = csv.writer(output, lineterminator="\n")
                writer.writerow(header)
                writer.writerows(data_rows)
                sample_csv = output.getvalue().strip()
                return sample_csv if sample_csv else None
        except Exception as e:
            last_error = e
            print(f"[WARN] sample fetch failed attempt={attempt}/{retries} url={download_url}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)
    print(f"[ERROR] giving up on url={download_url}: {last_error}")
    return None


def build_samples_schema() -> T.StructType:
    return T.StructType([
        T.StructField("dataset_id",           T.StringType(), True),
        T.StructField("source",               T.StringType(), True),
        T.StructField("title",                T.StringType(), True),
        T.StructField("original_description", T.StringType(), True),
        T.StructField("download_url",         T.StringType(), True),
        T.StructField("landing_page_url",     T.StringType(), True),
        T.StructField("sample_csv",           T.StringType(), True),
        T.StructField("sample_fetch_status",  T.StringType(), True),
        T.StructField("error_message",        T.StringType(), True),
    ])


def normalize_output_rows(output_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    required_cols = [
        "dataset_id", "source", "title", "original_description",
        "download_url", "landing_page_url", "sample_csv",
        "sample_fetch_status", "error_message",
    ]
    out_pd = pd.DataFrame(output_rows)
    for col in required_cols:
        if col not in out_pd.columns:
            out_pd[col] = None
    out_pd = out_pd[required_cols]
    for col in required_cols:
        out_pd[col] = out_pd[col].where(pd.notnull(out_pd[col]), None)
    return out_pd


In [ ]:
step1_df = spark.read.parquet(HDFS_METADATA)

metadata_pd = (
    step1_df
    .select("dataset_id", "source", "title", "original_description",
            "download_url", "landing_page_url")
    .toPandas()
)

output_rows: List[Dict[str, Any]] = []
total = len(metadata_pd)
print(f"[INFO] Total metadata rows to process: {total}")

for idx, row in metadata_pd.iterrows():
    dataset_id           = row.get("dataset_id")
    source               = row.get("source")
    title                = row.get("title")
    original_description = row.get("original_description")
    download_url         = row.get("download_url")
    landing_page_url     = row.get("landing_page_url")

    print(f"[INFO] Fetching sample {idx + 1}/{total} for source={source} dataset_id={dataset_id}")

    sample_csv    = None
    error_message = None
    status        = "error"

    try:
        if download_url is None or not str(download_url).strip():
            raise ValueError("missing_download_url")
        sample_csv = fetch_csv_sample_text(
            download_url=str(download_url),
            max_rows=MAX_SAMPLE_ROWS,
            request_timeout=REQUEST_TIMEOUT,
            max_elapsed_seconds=MAX_ELAPSED_SECONDS,
        )
        if sample_csv is None or not str(sample_csv).strip():
            raise ValueError("empty_or_unreadable_sample")
        status = "success"
    except Exception as e:
        error_message = str(e)
        print(f"[ERROR] source={source} dataset_id={dataset_id} error={error_message}")

    output_rows.append({
        "dataset_id":           None if dataset_id is None else str(dataset_id),
        "source":               None if source is None else str(source),
        "title":                None if title is None else str(title),
        "original_description": None if original_description is None else str(original_description),
        "download_url":         None if download_url is None else str(download_url),
        "landing_page_url":     None if landing_page_url is None else str(landing_page_url),
        "sample_csv":           sample_csv,
        "sample_fetch_status":  status,
        "error_message":        error_message,
    })

if not output_rows:
    raise ValueError("No output rows were produced.")

out_pd        = normalize_output_rows(output_rows)
samples_spark = spark.createDataFrame(out_pd, schema=build_samples_schema())
samples_spark.write.mode("overwrite").parquet(HDFS_SAMPLES)

print(f"[INFO] Wrote {len(out_pd)} sampled datasets to {HDFS_SAMPLES}")
samples_spark.groupBy("source", "sample_fetch_status").count().show(truncate=False)
samples_spark.select(
    "dataset_id", "source", "title", "sample_fetch_status", "error_message"
).show(100, truncate=False)


In [ ]:
step2_df = spark.read.parquet(HDFS_SAMPLES)

prepared_df = (
    step2_df
    .filter(F.col("sample_fetch_status") == "success")
    .filter(F.col("sample_csv").isNotNull())
    .filter(F.trim(F.col("sample_csv")) != "")
    .select(
        F.col("dataset_id").cast("string").alias("dataset_id"),
        F.col("title").cast("string").alias("title"),
        F.col("source").cast("string").alias("source"),
        F.col("sample_csv").cast("string").alias("sample_csv"),
        F.col("original_description").cast("string").alias("original_description"),
        F.col("download_url").cast("string").alias("download_url"),
        F.col("landing_page_url").cast("string").alias("landing_page_url"),
    )
)

prepared_df.write.mode("overwrite").parquet(HDFS_PREPARED)

print("[INFO] Prepared records written.")
print(f"[INFO] Output path: {HDFS_PREPARED}")
print(f"[INFO] Prepared row count: {prepared_df.count()}")
prepared_df.groupBy("source").count().show(truncate=False)
prepared_df.select("dataset_id", "source", "title").show(100, truncate=False)


In [ ]:
def run_cmd(cmd: List[str]) -> None:
    print(f"[CMD] {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


def shrink_sample_csv(
    sample_csv: Optional[str],
    max_chars: int = 8000,
    max_lines: int = 8,
) -> Optional[str]:
    if sample_csv is None:
        return None
    text = str(sample_csv).strip()
    if not text:
        return None
    lines      = text.splitlines()
    if not lines:
        return None
    header     = lines[0]
    data_lines = lines[1 : 1 + max_lines]
    shrunk     = "\n".join([header] + data_lines)
    if len(shrunk) > max_chars:
        shrunk = shrunk[:max_chars]
    return shrunk.strip() if shrunk.strip() else None


def copy_hdfs_parquet_to_local(hdfs_path: str, local_parent_dir: str) -> str:
    local_path = os.path.join(local_parent_dir, Path(hdfs_path).name)
    if os.path.exists(local_path):
        shutil.rmtree(local_path, ignore_errors=True)
    run_cmd(["hdfs", "dfs", "-get", hdfs_path, local_parent_dir])
    return local_path


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    local_input_parquet = copy_hdfs_parquet_to_local(HDFS_PREPARED, tmpdir)
    records_pd = pd.read_parquet(local_input_parquet)

    client  = OpenAI()
    autoddg = AutoDDG(client=client, model_name=MODEL_NAME)

    output_rows: List[Dict[str, Any]] = []
    total = len(records_pd)
    print(f"[INFO] Total prepared records to describe: {total}")

    for idx, row in records_pd.iterrows():
        dataset_id           = row.get("dataset_id")
        source               = row.get("source")
        title                = row.get("title")
        original_description = row.get("original_description")
        download_url         = row.get("download_url")
        landing_page_url     = row.get("landing_page_url")
        sample_csv           = row.get("sample_csv")

        print(f"[INFO] Generating description {idx + 1}/{total} for source={source} dataset_id={dataset_id}")

        generated_description = None
        generation_status     = "error"
        error_message         = None

        try:
            shrunk_sample_csv = shrink_sample_csv(
                sample_csv=sample_csv,
                max_chars=MAX_CHARS,
                max_lines=MAX_LINES,
            )
            if shrunk_sample_csv is None:
                raise ValueError("missing_or_empty_sample_csv_after_shrink")
            _, generated_description = autoddg.describe_dataset(
                dataset_sample=shrunk_sample_csv
            )
            generation_status = "success"
        except Exception as e:
            error_message = str(e)
            print(f"[ERROR] source={source} dataset_id={dataset_id} error={error_message}")

        output_rows.append({
            "dataset_id":            None if pd.isna(dataset_id) else str(dataset_id),
            "source":                None if pd.isna(source) else str(source),
            "title":                 None if pd.isna(title) else str(title),
            "original_description":  None if pd.isna(original_description) else str(original_description),
            "download_url":          None if pd.isna(download_url) else str(download_url),
            "landing_page_url":      None if pd.isna(landing_page_url) else str(landing_page_url),
            "sample_csv":            None if pd.isna(sample_csv) else str(sample_csv),
            "generated_description": generated_description,
            "generation_status":     generation_status,
            "error_message":         error_message,
        })

out_df      = pd.DataFrame(output_rows)
output_path = Path(LOCAL_OUTPUT_PARQUET)
output_path.parent.mkdir(parents=True, exist_ok=True)
out_df.to_parquet(output_path, index=False)

print(f"[INFO] Wrote {len(out_df)} rows locally to {output_path}")
print(out_df.groupby(["source", "generation_status"]).size().reset_index(name="count"))


In [ ]:
desc_df = spark.read.parquet(LOCAL_OUTPUT_PARQUET)

eval_df = (
    desc_df
    .withColumn(
        "has_original_description",
        F.when(
            F.col("original_description").isNotNull() &
            (F.trim(F.col("original_description")) != ""),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "original_description_len",
        F.length(F.coalesce(F.col("original_description"), F.lit("")))
    )
    .withColumn(
        "generated_description_len",
        F.length(F.coalesce(F.col("generated_description"), F.lit("")))
    )
    .select(
        "dataset_id",
        "source",
        "title",
        "has_original_description",
        "original_description",
        "generated_description",
        "original_description_len",
        "generated_description_len",
        "generation_status",
        "error_message",
    )
)

eval_df.write.mode("overwrite").parquet(HDFS_EVAL)

print("[INFO] Evaluation table written.")
print(f"[INFO] Output path: {HDFS_EVAL}")
print(f"[INFO] Row count: {eval_df.count()}")
eval_df.groupBy("source", "generation_status").count().show(truncate=False)
eval_df.groupBy("source", "has_original_description").count().show(truncate=False)
eval_df.select(
    "dataset_id", "source", "title",
    "original_description_len", "generated_description_len",
).show(50, truncate=False)


In [ ]:
results_pd = eval_df.toPandas()
successful = results_pd[results_pd["generation_status"] == "success"]

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Datasets requested           : {NYC_LIMIT + DATA_GOV_LIMIT}")
print(f"  NYC Open Data              : {NYC_LIMIT}")
print(f"  Data.gov                   : {DATA_GOV_LIMIT}")
print(f"Descriptions generated       : {len(successful)}")
print(f"Generation errors            : {len(results_pd) - len(successful)}")
print()
print("Generated description length stats:")
print(successful["generated_description_len"].describe().round(1).to_string())
print("=" * 60)


In [ ]:
sample_n = min(5, len(successful))
for _, row in successful.sample(n=sample_n, random_state=42).iterrows():
    print("─" * 60)
    print(f"Dataset : {row['title']}")
    print(f"Source  : {row['source']}")
    print()
    if row.get("original_description"):
        print("Original description:")
        print(f"  {str(row['original_description'])[:300]}")
        print()
    print("Generated description:")
    print(f"  {row['generated_description']}")
    print()


In [ ]:
spark.stop()
print("Spark session stopped.")
print(f"Final descriptions : {LOCAL_OUTPUT_PARQUET}")
print(f"Evaluation table   : {HDFS_EVAL}")
